In [142]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

In [141]:
df = pd.read_csv("Walmart_Sales.csv", index_col=0)
df

,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
Store,,,,,,,
1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106
...,...,...,...,...,...,...,...
45,28-09-2012,713173.95,0,64.88,3.997,192.013558,8.684
45,05-10-2012,733455.07,0,64.89,3.985,192.170412,8.667
45,12-10-2012,734464.36,0,54.47,4.000,192.327265,8.667


In [151]:
# Convert 'Date' column to datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

In [157]:
df = df.sort_values(by=['Store', 'Date'])

In [165]:
#Possible outliers, so lets clean them

Q1 = df['Weekly_Sales'].quantile(0.25)
Q3 = df['Weekly_Sales'].quantile(0.75)

IQR = Q3 - Q1

lb = Q1-1.5*IQR
ub = Q3 + 1.5*IQR

outliers= df[(df["Weekly_Sales"] < lb) | (df["Weekly_Sales"] > ub)].index# outlier value

In [166]:
clean_data = df.drop(outliers, axis = 0)
clean_data.shape

(5076, 9)

In [167]:
# Create lag features for 'Weekly_Sales' (useful for forecasting based on past sales)
df['lag_1'] = df.groupby('Store')['Weekly_Sales'].shift(1)
df['lag_2'] = df.groupby('Store')['Weekly_Sales'].shift(2)

In [168]:
df.dropna(inplace=True)

In [169]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
import numpy as np

# Select your features for multicollinearity testing
X = df[['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'lag_1', 'lag_2',]]

# Add a constant column for the intercept
X['Intercept'] = 1

# Calculate VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Drop the intercept if included
vif_data = vif_data[vif_data['Feature'] != 'Intercept']

# Display VIF values
print(vif_data)

        Feature        VIF
0   Temperature   1.091446
1    Fuel_Price   1.063169
2           CPI   1.228879
3  Unemployment   1.167776
4         lag_1  10.405475
5         lag_2  10.410728


/tmp/ipykernel_360/760695125.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['Intercept'] = 1


In [170]:
clean_data = df.drop(outliers, axis = 0)
clean_data.shape

(5004, 9)

In [171]:
# Select features for Ridge Regression
features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'lag_1', 'lag_2']
X = df[features]
y = df['Weekly_Sales']

In [172]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

In [173]:
# Standardize the feature variables (important for Ridge Regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [174]:
# Train the Ridge Regression model
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)

Ridge()

In [175]:
# Make predictions on the test set
y_pred = ridge_model.predict(X_test_scaled)

In [176]:
# Evaluate the model using Root Mean Squared Error (RMSE)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'Root Mean Squared Error (RMSE): {rmse}')

Root Mean Squared Error (RMSE): 114173.84229903377


In [177]:
# Optionally, view the first few predictions vs actual values
pred_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(pred_df.head())

          Actual      Predicted
Store                          
37     519255.68  526913.924494
37     513015.35  532811.164588
37     460020.74  530815.736745
37     515777.97  496590.541129
37     540189.70  511115.951455


In [178]:
from sklearn.model_selection import cross_val_score

# Perform cross-validation (with 5 folds)
cv_scores = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5, scoring='neg_mean_squared_error')

# Convert to positive and take the square root for RMSE
rmse_cv_scores = np.sqrt(-cv_scores)
print(f'Cross-validated RMSE scores: {rmse_cv_scores}')
print(f'Mean cross-validated RMSE: {np.mean(rmse_cv_scores)}')

Cross-validated RMSE scores: [172868.82804659 208159.80462339 170177.87742087 180658.35094042
 114002.24019193]
Mean cross-validated RMSE: 169173.4202446393


In [179]:
from sklearn.metrics import r2_score

# Calculate R-squared
r2 = r2_score(y_test, y_pred)
print(f'R-squared: {r2}')

R-squared: 0.914742650494194


In [180]:
from sklearn.model_selection import GridSearchCV

# Define a grid of alpha values
param_grid = {'alpha': [0.01, 0.1, 1.0, 10, 100]}
ridge_cv = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

# Best alpha value
print(f'Best alpha: {ridge_cv.best_params_}')

Best alpha: {'alpha': 0.01}


In [181]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Retrain Ridge Regression with the best alpha (0.1)
best_ridge_model = Ridge(alpha=0.1)
best_ridge_model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = best_ridge_model.predict(X_test_scaled)

# Evaluate the model
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'R-squared: {r2}')

RMSE: 114169.90809209162
MAE: 60522.679623506396
R-squared: 0.9147485259954573


In [182]:
param_grid = {'alpha': np.logspace(-4, 2, 100)}  # Alpha values between 0.0001 and 100
grid_search = GridSearchCV(Ridge(), param_grid, cv=5, scoring='neg_mean_squared_error')
grid_search.fit(X_train_scaled, y_train)

print(f'Best alpha: {grid_search.best_params_}')

Best alpha: {'alpha': 0.0001}


In [183]:
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Retrain Ridge Regression with the best alpha (0.1)
best_ridge_model = Ridge(alpha=0.0001)
best_ridge_model.fit(X_train_scaled, y_train)

# Make predictions on the test set
y_pred = best_ridge_model.predict(X_test_scaled)

# Evaluate the model
rmse = mean_squared_error(y_test, y_pred, squared=False)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'RMSE: {rmse}')
print(f'MAE: {mae}')
print(f'R-squared: {r2}')

RMSE: 114169.47428105802
MAE: 60520.445810830446
R-squared: 0.9147491738537312


In [184]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Oletetaan, että testidata on valmis
# Step 1: Laske Weekly_Sales-keskiarvo ja keskihajonta
mean_sales = y_test.mean()

print(f"Mean Weekly Sales: {mean_sales}")

# Step 2: Luodaan baseline-malli, joka ennustaa keskiarvon kaikille arvoille
baseline_predictions = np.full_like(y_test, mean_sales)

# Step 3: Laske baseline-mallin suorituskyky (RMSE, MAE, R-squared)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_predictions))
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_r2 = r2_score(y_test, baseline_predictions)

print(f"Baseline RMSE: {baseline_rmse}")
print(f"Baseline MAE: {baseline_mae}")
print(f"Baseline R-Squared: {baseline_r2}")

Mean Weekly Sales: 764653.2547402078
Baseline RMSE: 391021.5591571785
Baseline MAE: 321322.7989144671
Baseline R-Squared: 0.0
